# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source schema is provided via a Croissant JSON-LD URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will let us access details about available record sets and data fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers. We'll print out the main record set structure, and list field `@id`s so we can access data precisely.

> **Note:** All code uses only `@id` to reference record sets and fields, as recommended.

In [ ]:
# The dataset exposes record sets in dataset.metadata.record_sets
record_sets = getattr(metadata, 'record_sets', [])
print(f"Number of record sets: {len(record_sets)}")

for i, rs in enumerate(record_sets):
    print(f"[{i}] Record Set: name={rs.name}, @id={rs.id}")
    # List all Field @ids
    if hasattr(rs, 'fields'):
        print("Fields in this record set:")
        for field in rs.fields:
            print(f"  - name={field.name}, @id={field.id}, data type={field.data_type}")
    print("")

# For most Croissant datasets, there's a single main record set. Let's identify and keep the main record set id:
main_record_set_id = record_sets[0].id if len(record_sets) > 0 else None

## 3. Data Extraction
Let's load all records from a specific record set into a Pandas DataFrame for analysis. We'll use the record set `@id` and fields above.

> **Tip:** You must use the `@id` for all referencing, e.g. for `record_set` parameter in `dataset.records`.

In [ ]:
# Extract data from each available record set and load as DataFrame
dataframes = {}
for rs in record_sets:
    record_set_id = rs.id
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id}. DataFrame shape={dataframes[record_set_id].shape}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
    else:
        print(f"No records found for record set {record_set_id}.")

# For analysis below, pick the main record set
main_df = dataframes[main_record_set_id]
main_field_ids = main_df.columns.tolist()
print(f"Field (@id)s in main record set: {main_field_ids}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, such as filtering, normalizing, and grouping records. All columns/fields must be referenced by their `@id`, which you can retrieve from the earlier overview.

In [ ]:
# Example: Let's choose numeric fields
# Change these variables to the actual `@id` of numeric fields from your earlier display
# For demonstration, let's attempt to auto-select a numeric field by dtype
numeric_candidates = [fid for fid in main_df.columns if pd.api.types.is_numeric_dtype(main_df[fid])]
if len(numeric_candidates) == 0:
    # fallback: try to coerce numeric columns
    for fid in main_df.columns:
        coerced = pd.to_numeric(main_df[fid], errors='coerce')
        missing = coerced.isna().sum()
        if missing < len(main_df):
            main_df[fid+'_num'] = coerced
            numeric_candidates.append(fid+'_num')
            if len(numeric_candidates) == 1:
                break
if len(numeric_candidates) == 0:
    print('No numeric fields found for EDA.')
else:
    numeric_field_id = numeric_candidates[0]   # Use the first found numeric field
    print(f"Using numeric field: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean()  # Example: filter above the mean
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field, e.g. the first string/categorical field
    group_field = None
    for fid in main_df.columns:
        if fid != numeric_field_id and main_df[fid].dtype==object:
            group_field = fid
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Averages of {numeric_field_id} grouped by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships. We use `matplotlib` for basic charts, plotting on the numeric field(s) discovered above.


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if len(numeric_candidates) > 0:
    plt.figure(figsize=(7,4))
    main_df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If we grouped above, plot group averages
    if 'grouped_df' in globals():
        grouped_df.plot(x=group_field, y=numeric_field_id, kind='bar', figsize=(10,5))
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This exploration notebook demonstrated loading, reviewing, and analyzing the FAIR² mass spectrometry dataset defined by a Croissant schema and loaded with `mlcroissant`. All references to record sets and fields used their full Croissant `@id`s. For a more complete or in-depth analysis, consult the Croissant schema or documentation for complete field semantics, and adjust column selection as appropriate.